# DeepSeek OCR - Jupyter Notebook

Headless OCR ohne GUI. Passe `INPUT_FILE` an ein Bild oder PDF an und fuehre die Zellen der Reihe nach aus.

In [ ]:
from pathlib import Path
import sys

# Funktioniert unabhaengig davon, ob Jupyter aus dem Projekt- oder desktop-Ordner gestartet wurde.
REPO_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "dsocr").is_dir())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from dsocr.config import Settings, get_mode, PROMPTS
from dsocr.engines import create_engine
from dsocr.pipeline import resolve_prompt, run
from dsocr.preprocess.loader import load_document

INPUT_FILE = REPO_ROOT / "beispiel.png"
OUTPUT_DIR = REPO_ROOT / "dsocr-out"
BACKEND = "auto"  # "demo" fuer CPU/Tesseract, "auto" fuer das beste verfuegbare Backend
MODE = "gundam"
PROMPT = "markdown"

if not INPUT_FILE.is_file():
    raise FileNotFoundError(f"Eingabedatei nicht gefunden: {INPUT_FILE}")

In [ ]:
settings = Settings()
document = load_document(
    INPUT_FILE.name,
    INPUT_FILE.read_bytes(),
    dpi=settings.pdf_dpi,
    max_pages=settings.max_pages,
)
engine = create_engine(BACKEND, settings)
prompt = resolve_prompt(PROMPTS[PROMPT])

result = run(
    document.pages,
    engine,
    prompt,
    get_mode(MODE),
    settings=settings,
)
print(f"Seiten: {result.summary.pages}; Fehler: {result.summary.failed}")
print(f"Backend: {result.summary.backend}")
print(result.combined_markdown())

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "document.md").write_text(result.combined_markdown(), encoding="utf-8")
(OUTPUT_DIR / "document.txt").write_text(result.combined_text(), encoding="utf-8")
print(f"Ergebnis geschrieben nach: {OUTPUT_DIR}")